# 🚗 VehiclEye: Entrenamiento en Google Colab

**Duración total:** ~2 horas (con GPU gratis)

**Qué hace este notebook:**
1. Instala PyTorch + dependencias
2. Descarga ~1200 imágenes de vehículos desde Bing
3. Entrena EfficientNet-B0 (Transfer Learning)
4. Exporta a formato ONNX para Render
5. Descarga el modelo final

---

## 🔧 CELDA 1: Verificar GPU y montar Google Drive

In [ ]:
# Verificar GPU disponible
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
print("\n✓ GPU detectada y lista para usar")

# Montar Google Drive para guardar el modelo
from google.colab import drive
drive.mount('/content/gdrive')
print("\n✓ Google Drive montado")

## 📦 CELDA 2: Instalar dependencias

In [ ]:
# Instalar PyTorch (ya optimizado para GPU en Colab)
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q timm pillow requests bing-image-downloader

print("\n✓ PyTorch, timm, y dependencias instaladas")

# Verificar torch
import torch
print(f"✓ PyTorch version: {torch.__version__}")
print(f"✓ CUDA disponible: {torch.cuda.is_available()}")
print(f"✓ GPU: {torch.cuda.get_device_name(0)}")

## ⬇️ CELDA 3: Clonar repo VehiclEye y descargar imágenes

In [ ]:
# Clonar el repositorio (si no lo tienes)
import os
if not os.path.exists('/content/Electiva_3'):
    !git clone https://github.com/nick2331/Electiva_3.git /content/Electiva_3
    print("✓ Repositorio clonado")
else:
    print("✓ Repositorio ya existe")

os.chdir('/content/Electiva_3')

## 🖼️ CELDA 4: Descargar imágenes de vehículos (~15 minutos)

In [ ]:
from pathlib import Path
import subprocess
import sys

# Catálogo de vehículos
VEHICLE_CLASSES = [
    ("Toyota", "Corolla"), ("Toyota", "Hilux"),
    ("Chevrolet", "Spark"), ("Chevrolet", "Aveo"),
    ("Renault", "Logan"), ("Renault", "Sandero"), ("Renault", "Stepway"),
    ("Mazda", "3"), ("Mazda", "CX-5"),
    ("Hyundai", "Tucson"), ("Hyundai", "Accent"),
    ("Kia", "Picanto"), ("Kia", "Rio"),
    ("Nissan", "Frontier"), ("Nissan", "Versa"),
    ("Ford", "Fiesta"), ("Ford", "Escape"),
    ("Volkswagen", "Gol"), ("Volkswagen", "Jetta"),
    ("Suzuki", "Swift"),
]

output_dir = Path("ml/data/vehicleye_dataset")

def download_class(brand, model, count=50):
    """Descarga imágenes usando bing-image-downloader"""
    class_dir = output_dir / brand / model
    class_dir.mkdir(parents=True, exist_ok=True)
    
    existing = len(list(class_dir.glob("*.jpg")))
    if existing >= count:
        print(f"✓ {brand:12} {model:15} — {existing} imágenes (suficiente)")
        return
    
    print(f"⬇️  {brand:12} {model:15} — descargando...")
    try:
        from bing_image_downloader import downloader
        downloader.download(
            f"{brand} {model} car",
            limit=count,
            output_dir="dataset",
            adult_filter_off=True,
            force_replace=False,
            timeout=60,
            verbose=False,
        )
        # Mover archivos a la carpeta correcta
        import shutil
        temp = Path("dataset") / f"{brand}_{model}".replace(" ", "_")
        if temp.exists():
            for img in temp.glob("*.jpg"):
                shutil.move(str(img), str(class_dir / img.name))
            shutil.rmtree(temp, ignore_errors=True)
        print(f"✓ {brand:12} {model:15} — {count} imágenes descargadas")
    except Exception as e:
        print(f"⚠️  {brand:12} {model:15} — error: {str(e)[:50]}")

# Descargar todas las clases
print("\n📥 Descargando imágenes de vehículos...\n")
for brand, model in VEHICLE_CLASSES:
    download_class(brand, model, count=50)

# Contar total
total = sum(len(list((output_dir / b / m).glob("*.jpg"))) for b, m in VEHICLE_CLASSES)
print(f"\n✅ Total de imágenes descargadas: {total}")
print(f"   Directorio: {output_dir}")

## 🧠 CELDA 5: Entrenar el modelo (~1 hora con GPU)

In [ ]:
# Ejecutar entrenamiento
!python ml/train.py \
  --data_dir ml/data/vehicleye_dataset \
  --epochs_phase1 5 \
  --epochs_phase2 10 \
  --output ml/checkpoints/efficientnet_b0_vehicleye.pth

print("\n✅ Entrenamiento completado")

## 📊 CELDA 6: Evaluar el modelo (opcional, ~2 minutos)

In [ ]:
# Evaluar el modelo en el conjunto de validación
!python ml/evaluate.py

# Mostrar métricas
from pathlib import Path
metrics_file = Path("reports/model_metrics.txt")
if metrics_file.exists():
    print("\n📊 REPORTE DE MÉTRICAS:")
    print("="*60)
    print(metrics_file.read_text())
else:
    print("⚠️  No se generó el reporte (normal si hay errores)")

## 🔄 CELDA 7: Exportar a ONNX (formato Render) (~1 minuto)

In [ ]:
!python ml/export_onnx.py \
  --checkpoint ml/checkpoints/efficientnet_b0_vehicleye.pth \
  --output ml/checkpoints/vehicleye.onnx

# Verificar archivo
from pathlib import Path
onnx_file = Path("ml/checkpoints/vehicleye.onnx")
if onnx_file.exists():
    size_mb = onnx_file.stat().st_size / 1024 / 1024
    print(f"\n✅ Modelo ONNX exportado correctamente")
    print(f"   Archivo: vehicleye.onnx")
    print(f"   Tamaño: {size_mb:.1f} MB")
else:
    print("❌ Error: no se generó vehicleye.onnx")

## 💾 CELDA 8: Descargar modelo a tu computadora

In [ ]:
from google.colab import files
from pathlib import Path

# Descargar el modelo ONNX
onnx_path = Path("ml/checkpoints/vehicleye.onnx")
if onnx_path.exists():
    print("📥 Descargando vehicleye.onnx...")
    files.download(str(onnx_path))
    print("\n✅ Descarga completada. El archivo está en tu carpeta 'Descargas'.")
else:
    print("❌ El archivo vehicleye.onnx no existe")

# También descargar el checkpoint PyTorch (opcional)
pth_path = Path("ml/checkpoints/efficientnet_b0_vehicleye.pth")
if pth_path.exists():
    print("\n💡 El archivo .pth también está disponible para descargar si lo necesitas.")

## 📋 CELDA 9: Pasos siguientes

In [ ]:
print("""
🎉 ¡ENTRENAMIENTO COMPLETADO!

📦 Tienes el archivo: vehicleye.onnx (48-50 MB)

Próximos pasos:

1️⃣  Sube vehicleye.onnx a GitHub Releases:
    → https://github.com/nick2331/Electiva_3/releases
    → Click "Create a new release"
    → Tag: v1.0
    → Sube el archivo vehicleye.onnx

2️⃣  Copia la URL de descarga:
    → Haz clic derecho en vehicleye.onnx → Copy link
    → Debería ser algo como:
       https://github.com/nick2331/Electiva_3/releases/download/v1.0/vehicleye.onnx

3️⃣  Configura en Render:
    → https://dashboard.render.com
    → Servicio: vehicleye-api
    → Environment → Add Variable
    → Key: MODEL_DOWNLOAD_URL
    → Value: (pega la URL del paso 2)
    → Save

4️⃣  Render redeploya automáticamente. Verifica en:
    → /admin/health → model status debe decir "ok"

5️⃣  ¡Prueba en tu frontend!
    → Sube una foto de un Ford Raptor
    → Debería identificarlo como "Ford Raptor" (no random)

¿Preguntas? Pregunta en chat. 🚀
""")